# EGC Taxonomy Tree

Reads `.a.phylodist.txt` files from `Metagenome_Samples`, builds the taxonomy tree, and writes one `.tax.json` per sample.

**Run order:** Cell by cell top to bottom. Restart kernel before re-running.

In [1]:
import json
import time
from pathlib import Path
from collections import defaultdict
import sys
sys.path.append(r"C:\Users\sakoehn\Desktop\JGI\Notebooks")
from All_Coverages import allfold

## 1. Load sample list

In [2]:
with open(r'C:\Users\sakoehn\Desktop\JGI\sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f"Loaded {len(samples)} samples")

Loaded 128 samples


## 2. Read taxonomy annotation files → tax_db

`tax_db[gene_id]` holds parallel lists: one entry per observation of that gene.

The `lookupId` field is the full semicolon-separated taxonomy path from the last column of the phylodist file.

In [3]:
def read_tax(directpath, filename, s, tax_db):
    """
    Parse one sample's .a.phylodist.txt and append each gene observation
    to tax_db.  Both branches write to the same tax_db parameter.
    """
    path = directpath / filename
    if not path.exists():
        print(f"  WARNING: {path} not found, skipping")
        return

    with open(path, "r") as f:
        for line in f:
            parts   = line.rstrip("\n").split("\t")
            gene_id = parts[0]
            full_path = parts[-1]        # semicolon-separated taxonomy path
            scaff   = gene_id.rsplit('_', 2)[0]

            if gene_id not in tax_db:
                tax_db[gene_id] = {
                    'lookupId':  [full_path],
                    'timepoint': [s],
                    'scaffolds': [scaff],
                }
            else:
                tax_db[gene_id]['lookupId'].append(full_path)
                tax_db[gene_id]['timepoint'].append(s)
                tax_db[gene_id]['scaffolds'].append(scaff)


base   = Path(r"C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples")
tax_db = {}

for s in samples:
    read_tax(base / s, f"{s}.a.phylodist.txt", s, tax_db)

print(f"tax_db: {len(tax_db):,} genes")

tax_db: 71,121,876 genes


In [ ]:
# ── Gene-weighted depth denominator (preview) ───────────────────────────────
# For the corrected norm_egc formula we need, for each (sample, scaffold),
# the total number of annotated genes on that scaffold. Each gene is counted
# ONCE regardless of how many annotation terms it maps to (hierarchy nodes
# would otherwise inflate the count).
#
# gene_count_per_sample_scaff[sample][scaffold] = number of genes on that
# scaffold in that sample that appear in tax_db (the loaded annotation DB).
#
# NOTE: this is a preview/sanity-check only. avg_gene_map (below, in the
# "Compute EGC and norm_egc" section) does NOT read gene_count_per_sample_scaff
# — it recomputes its own weighted_total_per_sample from tax_db independently.
# Both count genes from the same tax_db so they agree, but this cell's result
# is never consumed downstream.

from collections import defaultdict as _dd

gene_count_per_sample_scaff = _dd(lambda: _dd(int))

for _gene_id, _ginfo in tax_db.items():
    _scaff = _gene_id.rsplit('_', 2)[0]
    for _sample in _ginfo.get('timepoint', []):
        gene_count_per_sample_scaff[_sample][_scaff] += 1

# Sanity check: print total gene-weighted depth for first sample
_first_sample = next(iter(gene_count_per_sample_scaff))
_covs = allfold.get(_first_sample, {})
_weighted = sum(
    gene_count_per_sample_scaff[_first_sample].get(s, 0) * c
    for s, c in _covs.items()
)
_plain = sum(_covs.values())
print(f'Sample {_first_sample}:')
print(f'  Old denominator (Σ AvgFold):            {_plain:,.0f}')
print(f'  New denominator (Σ gene_count×AvgFold): {_weighted:,.0f}')
print(f'  Ratio (new/old):                        {_weighted/_plain:.2f}x')
print(f'  → norm_egc values will be ~{_plain/_weighted:.2f}x smaller than before')


## 3. Build taxonomy tree

Flat dict keyed by taxonomy label. For each semicolon-separated path, every node in the path is registered and gets the gene + scaffold counted.

Fields per node: `depth` (1-indexed position in path), `category` (root label of the path), `parent` (direct parent label or `None` for roots).

Key = ID = name (they are identical for taxonomy labels).

In [5]:
tax_tree = {}  # {tax_id: {depth, category, parent, timepoints}}

start = time.time()

for gene_id, ginfo in tax_db.items():
    lookups    = ginfo.get('lookupId',  [])
    timepoints = ginfo.get('timepoint', [])
    scaffolds  = ginfo.get('scaffolds', [])

    for full_path, sample, scaff in zip(lookups, timepoints, scaffolds):
        raw_parts = full_path.split(';')
        taxpath   = [p.strip() for p in raw_parts if p.strip()]

        # JGI's phylodist format pads unresolved strain-level classifications
        # by repeating the deepest resolved rank's name as a placeholder
        # (e.g. "...;Lutibacter;Lutibacter sp. Hel_I_33_5;Lutibacter sp. Hel_I_33_5").
        # Without collapsing that, the loop below would visit the same node
        # twice for one gene observation and double-count it, inflating that
        # leaf's EGC — sometimes past its own parent's honestly-counted total.
        taxpath = [t for i, t in enumerate(taxpath) if i == 0 or t != taxpath[i - 1]]

        if not taxpath:
            continue

        root_id = taxpath[0]

        for i, tax_id in enumerate(taxpath):
            if tax_id not in tax_tree:
                tax_tree[tax_id] = {
                    'depth':      i + 1,
                    'category':   root_id,
                    'parent':     None if i == 0 else taxpath[i - 1],
                    'timepoints': {},
                }

            node = tax_tree[tax_id]

            if sample not in node['timepoints']:
                node['timepoints'][sample] = {
                    'egc':       0,
                    'norm_egc':  0.0,
                    'scaffolds': {},
                    'genes':     set(),
                }

            tp = node['timepoints'][sample]
            tp['genes'].add(gene_id)
            tp['scaffolds'][scaff] = tp['scaffolds'].get(scaff, 0) + 1

print(f"Taxonomy tree: {len(tax_tree):,} nodes  ({time.time() - start:.1f}s)")

Taxonomy tree: 60,235 nodes  (950.5s)


## 4. Compute EGC and norm_egc

`egc = Σ (gene_count_on_scaffold × scaffold_coverage)`

`norm_egc = egc / Σ (gene_count_on_scaffold × scaffold_coverage) for ALL scaffolds`

The denominator is gene-weighted total depth so norm_egc is always in [0, 1].

In [6]:
# ── Gene-weighted depth denominator (corrected norm_egc) ────────────────────
# norm_egc should be in [0, 1]: fraction of all annotated gene copies in this
# sample that belong to this term.
#
# Old denominator: allfold[sample]['total'] = Σ(AvgFold for all scaffolds)
#   → can give norm_egc > 1 for annotations on high-coverage scaffolds
#
# New denominator: Σ(gene_count × AvgFold for all scaffolds in sample)
#   → EGC[term] ≤ this sum by construction, so norm_egc ≤ 1

from collections import defaultdict as _dd

# Count genes per scaffold per sample from tax_db
_gene_counts = _dd(lambda: _dd(int))
for _gid, _ginfo in tax_db.items():
    _scaff = _gid.rsplit('_', 2)[0]
    for _s in _ginfo.get('timepoint', []):
        _gene_counts[_s][_scaff] += 1

# weighted_total_per_sample[sample] = Σ(gene_count × AvgFold)
weighted_total_per_sample = {}
for _s, _counts in _gene_counts.items():
    _covs = allfold.get(_s, {})
    weighted_total_per_sample[_s] = sum(
        _counts.get(sc, 0) * _covs.get(sc, 0)
        for sc in _covs if sc != 'total'
    )

# Sanity check on first sample
_s0 = next(iter(weighted_total_per_sample))
_new = weighted_total_per_sample[_s0]
_old = allfold.get(_s0, {}).get('total', 0)
print(f'Sample {_s0}: old denom={_old:,.0f}  new denom={_new:,.0f}')
print(f'norm_egc values will be ~{_old/_new:.2f}x smaller, now bounded by 1.0')


Sample 3300060358: old denom=8,298,642  new denom=15,595,564
norm_egc values will be ~0.53x smaller, now bounded by 1.0


In [7]:
def avg_gene_map(node_id):
    node = tax_tree[node_id]
    for sample_name, data in node['timepoints'].items():
        scaffold_counts = data.get('scaffolds')
        if not scaffold_counts or sample_name not in allfold:
            continue

        covs = allfold[sample_name]
        sample_total_depth = weighted_total_per_sample.get(
            sample_name, allfold[sample_name]['total']
        )

        total_egc = 0
        for scaff, cnt in scaffold_counts.items():
            cov = covs.get(scaff, 0)
            total_egc += cnt * cov

        data['egc']      = total_egc
        data['norm_egc'] = (
            total_egc / sample_total_depth
            if sample_total_depth > 0 else 0.0
        )


for node_id in tax_tree:
    avg_gene_map(node_id)
print('EGC computation complete')


EGC computation complete


## 5. Write .tax.json files

In [8]:
base = Path(r"C:\Users\sakoehn\Desktop\JGI\DataStructures\Taxonomy")
base.mkdir(parents=True, exist_ok=True)

for s in samples:
    tp_out = {}
    for tax_id, node in tax_tree.items():
        if s not in node['timepoints']:
            continue
        tp = node['timepoints'][s]
        tp_out[tax_id] = {
            'depth':     node['depth'],
            'category':  node['category'],
            'parent':    node['parent'],
            'egc':       tp.get('egc',       0),
            'norm_egc':  tp.get('norm_egc',  0.0),
            'scaffolds': tp.get('scaffolds', {}),
        }
    with open(base / f"{s}.tax.json", 'w') as f:
        json.dump(tp_out, f, indent=2)

print(f"Wrote {len(samples)} .tax.json files to {base}")

Wrote 128 .tax.json files to C:\Users\sakoehn\Desktop\JGI\DataStructures\Taxonomy
